# Notebook 03: Structured Output & Schema-Constrained Generation

Companion to Module 03. Real experiments against live `gpt-4o-mini`:
1. A fair, same-input, three-way comparison — JSON mode vs. structured outputs vs. function calling — measuring real schema validity, parsing failures, retries, latency, and tokens.
2. A real validation-retry pipeline with the real distribution of attempts-to-success, not just the average.

In [1]:
import os
import json
import time
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError

load_dotenv(find_dotenv())
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"

class PersonInfo(BaseModel):
    name: str
    age: int
    occupation: str
    is_employed: bool

print(f"OpenAI client ready. Model: {MODEL}. Schema: PersonInfo(name, age, occupation, is_employed)")

OpenAI client ready. Model: gpt-4o-mini. Schema: PersonInfo(name, age, occupation, is_employed)


## 1. Fair Three-Way Comparison: JSON Mode vs. Structured Outputs vs. Function Calling

The SAME 6 real, deliberately edge-case-heavy bios (missing age, ambiguous employment status) run through all three mechanisms. Every raw result is re-validated against the real Pydantic schema in application code -- never trusting provider-side enforcement alone.

In [2]:
BIOS = [
    "Maria Gonzalez, 34, works as a senior software engineer at a fintech startup.",
    "Long-retired postal worker James Whitfield just celebrated his 71st birthday last week.",
    "Meet Aisha, a freelance graphic designer currently between contracts.",  # age NOT stated -- deliberate edge case
    "Tom, 29 years young, spends his days coding open source projects for free and doesn't have a paying job right now.",
    "Dr. Elena Petrova (52) leads the oncology department and still occasionally teaches at the medical school.",
    "Unemployed since the layoffs, 38-year-old Marcus spends most of his time job hunting and doing occasional consulting gigs.",  # ambiguous employment
]

EXTRACT_INSTRUCTION = (
    "Extract person information from the bio as JSON with EXACTLY these fields: "
    "name (string), age (integer), occupation (string), is_employed (boolean). "
    "If age is not stated, make your best real integer estimate from context -- never omit the field."
)

TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "extract_person_info",
        "description": "Extract structured person information from a bio.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "integer"},
                "occupation": {"type": "string"},
                "is_employed": {"type": "boolean"},
            },
            "required": ["name", "age", "occupation", "is_employed"],
        },
    },
}

def try_validate(raw_dict):
    try:
        PersonInfo(**raw_dict)
        return True, None
    except ValidationError as e:
        return False, str(e)[:150]

def run_json_mode(bio):
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.0, response_format={"type": "json_object"},
        messages=[{"role": "system", "content": EXTRACT_INSTRUCTION}, {"role": "user", "content": bio}],
    )
    latency_ms = (time.perf_counter() - start) * 1000
    try:
        raw = json.loads(resp.choices[0].message.content)
    except json.JSONDecodeError:
        return False, "invalid JSON syntax", latency_ms, resp.usage.total_tokens
    valid, err = try_validate(raw)
    return valid, err, latency_ms, resp.usage.total_tokens

def run_structured_outputs(bio):
    start = time.perf_counter()
    resp = client.chat.completions.parse(
        model=MODEL, temperature=0.0,
        messages=[{"role": "system", "content": EXTRACT_INSTRUCTION}, {"role": "user", "content": bio}],
        response_format=PersonInfo,
    )
    latency_ms = (time.perf_counter() - start) * 1000
    parsed = resp.choices[0].message.parsed
    if parsed is None:
        return False, "provider refused/failed to parse", latency_ms, resp.usage.total_tokens
    valid, err = try_validate(parsed.model_dump())
    return valid, err, latency_ms, resp.usage.total_tokens

def run_function_calling(bio):
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.0, tools=[TOOL_SCHEMA],
        tool_choice={"type": "function", "function": {"name": "extract_person_info"}},
        messages=[{"role": "system", "content": EXTRACT_INSTRUCTION}, {"role": "user", "content": bio}],
    )
    latency_ms = (time.perf_counter() - start) * 1000
    tool_calls = resp.choices[0].message.tool_calls
    if not tool_calls:
        return False, "no tool call returned", latency_ms, resp.usage.total_tokens
    try:
        raw = json.loads(tool_calls[0].function.arguments)
    except json.JSONDecodeError:
        return False, "invalid JSON in tool arguments", latency_ms, resp.usage.total_tokens
    valid, err = try_validate(raw)
    return valid, err, latency_ms, resp.usage.total_tokens

def run_mechanism(fn, label):
    valid_count, total_tokens, total_latency = 0, 0, 0.0
    failures = []
    for bio in BIOS:
        valid, err, latency_ms, tokens = fn(bio)
        valid_count += int(valid)
        total_tokens += tokens
        total_latency += latency_ms
        if not valid:
            failures.append((bio[:40], err))
    print(f"=== {label} ===")
    print(f"Schema-valid: {valid_count}/{len(BIOS)}  Tokens: {total_tokens}  Latency: {total_latency:.1f}ms")
    for bio_snip, err in failures:
        print(f"  FAILURE: {bio_snip}... -> {err}")
    return valid_count, total_tokens, total_latency

json_valid, json_tokens, json_latency = run_mechanism(run_json_mode, "JSON MODE")
so_valid, so_tokens, so_latency = run_mechanism(run_structured_outputs, "STRUCTURED OUTPUTS")
fc_valid, fc_tokens, fc_latency = run_mechanism(run_function_calling, "FUNCTION CALLING")

print(f"\nSummary (validity/6, tokens, latency-ms):")
print(f"  JSON mode:          {json_valid}/6, {json_tokens}, {json_latency:.0f}")
print(f"  Structured outputs: {so_valid}/6, {so_tokens}, {so_latency:.0f}")
print(f"  Function calling:   {fc_valid}/6, {fc_tokens}, {fc_latency:.0f}")

=== JSON MODE ===
Schema-valid: 6/6  Tokens: 703  Latency: 7847.3ms


=== STRUCTURED OUTPUTS ===
Schema-valid: 6/6  Tokens: 1068  Latency: 7449.1ms


=== FUNCTION CALLING ===
Schema-valid: 6/6  Tokens: 983  Latency: 5817.0ms

Summary (validity/6, tokens, latency-ms):
  JSON mode:          6/6, 703, 7847
  Structured outputs: 6/6, 1068, 7449
  Function calling:   6/6, 983, 5817


### Output Explanation: Three-Way Structured-Output Comparison

An honest, real negative result on validity: all three mechanisms scored a perfect `6/6` schema-valid, including on the deliberately tricky bios (Aisha's missing age, Marcus's ambiguous "unemployed... occasional consulting" status). `gpt-4o-mini` handled every edge case correctly enough to pass Pydantic validation regardless of mechanism — this specific model, on this specific task, wasn't hard enough to surface the validity-rate gap the module's theory predicts *can* exist between weaker and stronger guarantees. That's a genuinely useful real finding in its own right: don't assume a validity-rate difference will always show up — measure it, and when it doesn't, look at the *other* real dimensions.

And the other dimensions did differentiate, clearly: **Structured outputs used the most tokens** (`1068`, a real `+52.0%` over JSON mode's `703`) despite offering the strongest schema guarantee — real evidence that provider-side schema enforcement isn't free, it costs real additional tokens (likely schema-description overhead baked into the request). **Function calling was both cheaper than structured outputs (`983` vs `1068` tokens) and the fastest of all three** (`5817.0ms`, real `-25.9%` vs JSON mode's `7847.3ms` and `-21.9%` vs structured outputs' `7449.1ms`). The real, production-relevant takeaway: when validity rates tie, function calling was the cheapest way to get a real schema guarantee here — but per Module 03's own distinction, function calling is semantically for invoking an action, while structured outputs is semantically for direct data extraction; a real system should pick based on that intent, not purely on this one real cost comparison, even though the cost comparison is a real, legitimate factor to weigh.

## 2. Real Validation-Retry Pipeline: Distribution of Attempts, Not Just the Average

Using JSON mode (the weakest real-measured condition above) with a real repair-retry loop: on validation failure, the real Pydantic error is fed back into a real repair call, up to 3 real attempts per bio. Recorded as a real distribution -- 1 attempt / 2 attempts / 3+ attempts -- across all 6 bios, not collapsed into a single average.

In [3]:
def run_json_mode_with_history(messages):
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.0, response_format={"type": "json_object"}, messages=messages,
    )
    latency_ms = (time.perf_counter() - start) * 1000
    return resp.choices[0].message.content, latency_ms, resp.usage.total_tokens

def repair_pipeline(bio, max_attempts=3):
    messages = [
        {"role": "system", "content": EXTRACT_INSTRUCTION},
        {"role": "user", "content": bio},
    ]
    for attempt in range(1, max_attempts + 1):
        raw_text, latency_ms, tokens = run_json_mode_with_history(messages)
        try:
            raw = json.loads(raw_text)
            PersonInfo(**raw)
            return True, attempt
        except (json.JSONDecodeError, ValidationError) as e:
            messages.append({"role": "assistant", "content": raw_text})
            messages.append({"role": "user", "content": f"That output failed validation with error: {str(e)[:200]}. Please return ONLY corrected JSON matching the schema exactly."})
    return False, max_attempts

attempt_results = []
for bio in BIOS:
    success, attempts = repair_pipeline(bio)
    attempt_results.append((bio[:40], success, attempts))
    print(f"[{'SUCCESS' if success else 'EXHAUSTED'}] attempts={attempts} | {bio[:40]}...")

from collections import Counter
dist = Counter(a for _, success, a in attempt_results if success)
failed_after_max = sum(1 for _, success, a in attempt_results if not success)

print(f"\nReal attempt distribution across {len(BIOS)} bios:")
print(f"  1 attempt:  {dist.get(1, 0)}")
print(f"  2 attempts: {dist.get(2, 0)}")
print(f"  3+ attempts (succeeded on final try): {dist.get(3, 0)}")
print(f"  Exhausted (never succeeded within 3): {failed_after_max}")
avg_attempts = sum(a for _, s, a in attempt_results if s) / max(1, sum(1 for _, s, a in attempt_results if s))
print(f"\nFor reference, the average alone would have reported: {avg_attempts:.2f} attempts -- the distribution above is the more informative real signal.")

[SUCCESS] attempts=1 | Maria Gonzalez, 34, works as a senior so...


[SUCCESS] attempts=1 | Long-retired postal worker James Whitfie...


[SUCCESS] attempts=1 | Meet Aisha, a freelance graphic designer...


[SUCCESS] attempts=1 | Tom, 29 years young, spends his days cod...


[SUCCESS] attempts=1 | Dr. Elena Petrova (52) leads the oncolog...


[SUCCESS] attempts=1 | Unemployed since the layoffs, 38-year-ol...

Real attempt distribution across 6 bios:
  1 attempt:  6
  2 attempts: 0
  3+ attempts (succeeded on final try): 0
  Exhausted (never succeeded within 3): 0

For reference, the average alone would have reported: 1.00 attempts -- the distribution above is the more informative real signal.


### Output Explanation: Real Retry-Attempt Distribution

The real distribution was `1 attempt: 6, 2 attempts: 0, 3+ attempts: 0, Exhausted: 0` — every single bio succeeded on the first real attempt, so the average (`1.00`) and the distribution tell the same story here. This is a direct, consistent consequence of Section 1's own finding: since JSON mode already achieved `6/6` real validity on this exact bio set moments earlier, re-running the identical task through the repair pipeline unsurprisingly needed zero real repairs — the model's real performance on this specific task didn't change between runs.

This real run is honest about what it does and doesn't demonstrate: it confirms the repair pipeline's plumbing executes correctly end-to-end against a real API (message history construction, real JSON parsing, real Pydantic validation), but it did **not** get to exercise the actual repair path, since no real failure ever occurred to repair. The repair mechanism's correctness under a genuine failure (the specific behavior of feeding a real validation error back into a real follow-up call) was verified separately in Module 03's own Track 1 reference code via a mock generator deliberately designed to fail its first attempt — this notebook and that mock-based test are complementary: one proves the pipeline works against a real API on a real task, the other proves the repair branch itself functions correctly when a failure genuinely occurs.

## 3. Cleanup

In [4]:
del client
print("Real OpenAI client released. This notebook used no local GPU model, so no CUDA cleanup is needed.")

Real OpenAI client released. This notebook used no local GPU model, so no CUDA cleanup is needed.
